# 08 Evaluation

This notebook consolidates model prediction outputs into discrimination, calibration, confusion-matrix, and clinical-utility artifacts. It is designed to support both baseline and multimodal models through a common prediction-table format.

## Evaluation scope

- AUROC and AUPRC
- Precision, recall, F1, sensitivity, specificity
- Brier score and calibration tables
- ROC and PR curve points
- Lead-time summaries by horizon
- Reusable CSV artifacts for figure generation

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/multimodal-early-sepsis') if IN_COLAB else Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
if IN_COLAB:
    %pip -q install pyyaml pandas numpy scikit-learn

In [ ]:
import pandas as pd

from src.evaluation.artifacts import build_lead_time_table, summarize_predictions
from src.utils.config import load_config
from src.utils.io_utils import save_dataframe_bundle
from src.utils.logging_utils import write_run_manifest
from src.utils.paths import ensure_directories, resolve_project_paths

In [ ]:
base_config_path = PROJECT_ROOT / 'configs' / 'base.yaml'
override_config_path = PROJECT_ROOT / 'configs' / 'colab.yaml' if IN_COLAB else None
config = load_config(base_config_path, override_config_path)
paths = resolve_project_paths(config)
ensure_directories(paths)
config['evaluation']

## Collect prediction tables

By default this notebook evaluates baseline prediction artifacts from Notebook 06. Later, multimodal prediction tables can be dropped into the same directory pattern and evaluated with the same logic.

In [ ]:
baseline_dir = paths['processed_data_dir'] / '06_baseline_models'
prediction_files = sorted(baseline_dir.glob('*_test_predictions.csv'))
assert prediction_files, 'No baseline prediction files found. Run 06_baseline_models first.'
prediction_files[:10]

## Evaluate each prediction file

In [ ]:
metric_rows = []
artifact_bundle = {}

for path in prediction_files:
    predictions_df = pd.read_csv(path)
    model_name = predictions_df['model_name'].iloc[0] if 'model_name' in predictions_df.columns else path.stem
    dataset_name = predictions_df['dataset_name'].iloc[0] if 'dataset_name' in predictions_df.columns else 'unknown_dataset'
    horizon_hours = int(dataset_name.replace('horizon_', '').replace('h', '')) if dataset_name.startswith('horizon_') else -1

    metrics, curves = summarize_predictions(
        predictions_df=predictions_df,
        threshold=config['evaluation']['default_threshold'],
        calibration_bins=config['evaluation']['calibration_bins'],
    )
    metric_rows.append({
        'dataset_name': dataset_name,
        'model_name': model_name,
        'split': 'test',
        **metrics,
    })

    lead_time_df = build_lead_time_table(predictions_df, horizon_hours=horizon_hours)
    artifact_bundle[f'{dataset_name}_{model_name}_roc_curve'] = curves['roc_curve']
    artifact_bundle[f'{dataset_name}_{model_name}_pr_curve'] = curves['pr_curve']
    artifact_bundle[f'{dataset_name}_{model_name}_calibration'] = curves['calibration']
    artifact_bundle[f'{dataset_name}_{model_name}_lead_time'] = lead_time_df

evaluation_df = pd.DataFrame(metric_rows).sort_values(['dataset_name', 'model_name']).reset_index(drop=True)
evaluation_df

## Inspect strongest test results

In [ ]:
evaluation_df.sort_values('auprc', ascending=False).head(12) if not evaluation_df.empty else evaluation_df

## Save evaluation artifacts

In [ ]:
output_dir = paths['processed_data_dir'] / '08_evaluation'
artifact_bundle['evaluation_summary'] = evaluation_df
saved_paths = save_dataframe_bundle(artifact_bundle, output_dir)
saved_paths

In [ ]:
manifest_path = paths['manifests_dir'] / '08_evaluation_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='08_evaluation',
    config=config,
    extra={
        'saved_artifacts': saved_paths,
        'prediction_files_evaluated': [str(path) for path in prediction_files],
        'num_models_evaluated': int(evaluation_df['model_name'].nunique()) if not evaluation_df.empty else 0,
    },
)
manifest_path

## Review checklist

Before ablation analysis, review:

- Which model wins on AUROC versus AUPRC?
- Are calibration tables noticeably misaligned?
- Does performance degrade as horizon length increases?
- Are sensitivity and specificity balanced enough for clinical use?
- Which prediction files should be carried into the paper figures?

## Next notebook

`09_ablation_experiments.ipynb` will compare modality subsets, temporal feature subsets, and fusion strategies to strengthen the research contribution.